In [ ]:
# Cell 1: Imports and Environment Setup
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.options.plotting.backend = "plotly"

# Visual Template
TEMPLATE = "plotly_white"

# --- PATH CONFIGURATION ---
# Automatically detects project root
CURRENT_PATH = Path.cwd()
if CURRENT_PATH.name == 'notebooks':
    PROJECT_ROOT = CURRENT_PATH.parent
else:
    PROJECT_ROOT = CURRENT_PATH

# CORRECTION HERE: Pointing to where your file actually is (processed)
DATA_DIR = PROJECT_ROOT / "data" / "processed" 
OUTPUT_DIR = PROJECT_ROOT / "reports" / "figures"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Reading data from: {DATA_DIR}")

In [ ]:
# Cell 2: Data Loading and Consolidation (Ensuring 31 Million Rows)
import glob

try:
    # 1. Try to find partial processed files (Fendt 722_processed.parquet, etc.)
    # This is safer than relying on a single 'full' file which might be outdated
    part_files = list(DATA_DIR.glob("*_processed.parquet"))
    
    if not part_files:
        print("Warning: No partial files (*_processed.parquet) found.")
        print("Attempting to load telemetry_full.parquet directly...")
        df = pd.read_parquet(DATA_DIR / "telemetry_full.parquet")
    else:
        print(f"Found {len(part_files)} processed files. Consolidating now...")
        for f in part_files:
            print(f" -> {f.name}")
            
        # Reads and concatenates everything at once
        df = pd.concat([pd.read_parquet(f) for f in part_files], ignore_index=True)
        
    # --- DATA ENGINEERING (Metrics Calculation) ---
    print("\nCalculating temporal metrics (Delta-T)...")
    
    # Sorting is mandatory for diff() to work correctly
    # We sort by Tractor -> Source File -> Time
    df.sort_values(by=['tractor', 'source_file', 'timestamp'], inplace=True)
    
    # Calculate time difference between consecutive rows
    df['delta_t'] = df.groupby(['tractor', 'source_file'])['timestamp'].diff().fillna(0)
    
    # CUTOFF FILTER (Important for hours calculation)
    # If interval > 300s (5 min), we assume the tractor was turned off or logger stopped.
    # We do not count this as "operational hour".
    df.loc[df['delta_t'] > 300, 'delta_t'] = 0
    df.loc[df['delta_t'] < 0, 'delta_t'] = 0 # Protection against sorting errors
    
    # Recalculate consumption (L/h * h)
    df['liters_consumed'] = (df['fuel_rate'] * df['delta_t']) / 3600.0

    # --- FINAL LOADING REPORT ---
    total_recs = len(df)
    total_hours = df['delta_t'].sum() / 3600
    
    print("-" * 50)
    print(f"LOADING COMPLETED SUCCESSFULLY!")
    print(f"Total Records: {total_recs:,.0f}")
    print(f"Total Calculated Hours: {total_hours:,.2f} h")
    print("-" * 50)
    
    if total_recs < 30000000:
        print("⚠️ ALERT: Total records is still under 30 million.")
        print("Check if all 'Fendt XXX_processed.parquet' files are in the folder.")
        
except Exception as e:
    print(f"CRITICAL ERROR: {e}")

In [ ]:
# Cell 3: Fleet Descriptive Table (The "Big Picture")

# Aggregation by Tractor
summary = df.groupby('tractor').agg({
    'delta_t': 'sum',            # Sum of time in seconds
    'liters_consumed': 'sum',    # Sum of fuel
    'source_file': 'nunique',    # Count of missions
    'veh_speed': 'mean',         # Global average speed
    'fuel_rate': 'mean'          # Global average fuel rate
}).reset_index()

# Unit Conversion
summary['Total Hours (h)'] = (summary['delta_t'] / 3600).round(2)
summary['Total Diesel (L)'] = summary['liters_consumed'].round(2)
summary['Avg Consumption (L/h)'] = (summary['Total Diesel (L)'] / summary['Total Hours (h)']).round(2)
summary['Missions'] = summary['source_file']

# Selection and Renaming for Thesis Format
final_table = summary[['tractor', 'Missions', 'Total Hours (h)', 'Total Diesel (L)', 'Avg Consumption (L/h)']]
final_table.columns = ['Tractor', 'No. Missions', 'Operational Hours', 'Total Consumption (L)', 'Average (L/h)']

print("=== OPERATIONAL SUMMARY OF THE SEASON ===")
# Display with gradient to visually highlight highest values
display(final_table.style.background_gradient(cmap='Blues', subset=['Operational Hours', 'Total Consumption (L)']))

# General Totals for text
total_h = summary['Total Hours (h)'].sum()
total_l = summary['Total Diesel (L)'].sum()
print(f"\nFLEET TOTAL: {total_h:,.2f} hours analyzed | {total_l:,.2f} liters consumed.")

In [ ]:
# Inspection Cell (can be deleted later)
unique_activities = df['activity'].unique()
unique_activities.sort()

print("Copy and paste this into your 'translations' dictionary:\n")
for act in unique_activities:
    print(f"    '{act}': '{act}', # <--- Translate here")

In [ ]:
# Cell 4: Activity Distribution (Interactive Treemap) with 3-Line Footer

# 1. Data Preparation
tree_data = df.groupby(['tractor', 'activity'])[['delta_t', 'liters_consumed']].sum().reset_index()
tree_data['Hours'] = tree_data['delta_t'] / 3600

# 2. Definition of Labels (Renaming/Cleaning)
# I changed 'not working' to 'Idle' to make the map useful.
activity_labels = {
    'Cultivating (deep)': 'Cultivating (deep)',
    'Cultivating (shallow)': 'Cultivating (shallow)',
    'Disc harrowing': 'Disc harrowing',
    'Fertilizing': 'Fertilizing',
    'Mowing (front)': 'Mowing (front)',
    'Mowing (large-scale)': 'Mowing (large-scale)',
    'Mulching': 'Mulching',
    'Ploughing': 'Ploughing',
    'Power harrowing': 'Power harrowing',
    'Precision air seeding': 'Precision air seeding',
    'Rotary tilling': 'Rotary tilling',
    'Seed drill combination': 'Seed drill combination',
    'Seedbed combination': 'Seedbed combination',
    'Spraying': 'Spraying',
    'Swathing': 'Swathing',
    'Transport': 'Transport',
    'not working': 'Idle' # Improved English term
}

# --- 3-LINE BREAK LOGIC ---
# Filter only what exists in the data to avoid clutter
valid_items = [f"<i>{k}</i> = {v}" for k, v in activity_labels.items() if k in tree_data['activity'].unique()]

# Mathematical calculation to split into 3 equal parts
n = len(valid_items)
chunk = (n // 3) + 1 # Size of each chunk

line1 = ", ".join(valid_items[:chunk])
line2 = ", ".join(valid_items[chunk:chunk*2])
line3 = ", ".join(valid_items[chunk*2:])

# Assembles text with 3 line breaks (<br>)
footer_text = f"<b>Activity Key:</b><br>{line1}<br>{line2}<br>{line3}"

# 3. Apply labels directly to the column
tree_data['activity_label'] = tree_data['activity'].map(activity_labels).fillna(tree_data['activity'])

# 4. Chart Generation
fig = px.treemap(
    tree_data, 
    path=[px.Constant("Fendt Fleet"), 'tractor', 'activity_label'], 
    values='Hours',
    color='liters_consumed',
    color_continuous_scale='RdBu_r', 
    title="<b>Season Time Distribution:</b> Size = Hours, Color = Total Consumption (L)",
    hover_data={'Hours':':.2f', 'liters_consumed':':.2f'}
)

# 5. Layout Adjustment (Margin for 3 lines)
fig.update_layout(
    template=TEMPLATE,
    # Increased bottom margin (b) to 160px to fit the footer
    margin=dict(b=160, l=20, r=20, t=50),
    annotations=[
        dict(
            x=0, 
            y=-0.25,          # Lower down to avoid touching the chart
            xref="paper", 
            yref="paper",
            text=footer_text,
            showarrow=False,
            font=dict(size=10, color="grey"),
            align="left",
            xanchor="left",
            yanchor="top"
        )
    ]
)

fig.show()

In [ ]:
# ==============================================================================
# CELL 5 (UPDATED): OPERATIONAL PROFILE WITH CONSISTENT COLORS
# ==============================================================================
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Target DataFrame Definition
if 'full_df' in locals():
    target_df = full_df
else:
    print("Warning: 'full_df' not found. Attempting to use 'df'...")
    target_df = df

print("Generating comparative histograms (Unified Colors)...")

# Sampling
SAMPLE_N = 100000
if len(target_df) > SAMPLE_N:
    sample_df = target_df.sample(n=SAMPLE_N, random_state=42)
else:
    sample_df = target_df.copy()

# Subplots
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("<b>Speed Distribution (m/s)</b>", 
                    "<b>RPM Distribution (RPM)</b>")
)

tractors = sample_df['tractor'].unique()

# --- COLOR PALETTE ---
# We pick a vibrant Plotly palette to ensure distinction
palette = px.colors.qualitative.Bold 

for i, tractor in enumerate(tractors):
    subset = sample_df[sample_df['tractor'] == tractor]
    
    # Select a fixed color for this tractor (recycling if list runs out)
    tractor_color = palette[i % len(palette)]
    
    # 1. Speed
    speed_data = subset[(subset['veh_speed'] > 0.1) & (subset['veh_speed'] < 15)]['veh_speed']
    fig.add_trace(
        go.Histogram(
            x=speed_data, 
            name=f"{tractor}",
            nbinsx=80,
            histnorm='percent', 
            opacity=0.6,
            marker_color=tractor_color,  # <--- FIXED COLOR HERE
            legendgroup=tractor
        ),
        row=1, col=1
    )

    # 2. RPM
    rpm_data = subset[subset['engine_rpm'] > 100]['engine_rpm']
    fig.add_trace(
        go.Histogram(
            x=rpm_data, 
            name=f"{tractor}",
            nbinsx=80,
            histnorm='percent', 
            opacity=0.6,
            marker_color=tractor_color,  # <--- SAME COLOR HERE
            legendgroup=tractor,
            showlegend=False
        ),
        row=1, col=2
    )

# --- THRESHOLDS ---
THRESHOLD_SPEED = 1.0  
THRESHOLD_RPM = 1000   

fig.add_vline(x=THRESHOLD_SPEED, line_width=2, line_dash="dash", line_color="black", 
              annotation_text="Idle Cutoff", annotation_position="top right", row=1, col=1)

fig.add_vline(x=THRESHOLD_RPM, line_width=2, line_dash="dash", line_color="black", 
              annotation_text="Idle Cutoff", annotation_position="top right", row=1, col=2)

fig.update_layout(
    title_text="<b>Operational Profile</b>",
    barmode='overlay', 
    height=500,
    template="plotly_white",
    xaxis_title="Speed (m/s)",
    yaxis_title="Frequency (%)"
)
fig.update_yaxes(title_text="Frequency (%)", row=1, col=2)
fig.update_xaxes(title_text="Engine Speed (RPM)", row=1, col=2)

fig.show()

In [ ]:
# Cell 6: Operation Map (RPM vs Consumption)

# Sample of 50k points for performance/fluidity
scatter_sample = df.sample(n=50000, random_state=22)

fig = px.scatter(
    scatter_sample, 
    x='engine_rpm', 
    y='fuel_rate', 
    color='tractor', 
    size='veh_speed', # Bubble size = Speed
    size_max=12,
    opacity=0.6,
    title="<b>Efficiency Map:</b> RPM vs Consumption (Size = Speed)",
    labels={'engine_rpm': 'Engine Speed (RPM)', 'fuel_rate': 'Consumption (L/h)'},
    hover_data=['activity']
)

fig.update_layout(template=TEMPLATE)
fig.show()

In [ ]:
# Cell 7: Event-Based Classification (Hybrid: Speed + Duration)

print("--- Applying Event-Based Classification ---")

# 1. PARAMETERS (Your Logic)
SPEED_THRESHOLD = 0.75  # 2.7 km/h (More permissive on speed)
RPM_ON_THRESHOLD = 100
PTO_ACTIVE_THRESHOLD = 150
MIN_IDLE_DURATION = 0     # Seconds (Only counts as idle if stopped for > X sec)

# 2. Instant Classification (Idle Candidates)
# First, we mark everything that "looks like" idle
mask_off = df['engine_rpm'] < RPM_ON_THRESHOLD
mask_low_speed = df['veh_speed'] < SPEED_THRESHOLD
mask_pto_off = df['pto_rpm'].fillna(0) < PTO_ACTIVE_THRESHOLD

# Provisional State
df['state_raw'] = 'Working'
df.loc[mask_off, 'state_raw'] = 'Off'
df.loc[(~mask_off) & mask_low_speed & mask_pto_off, 'state_raw'] = 'Idle_Candidate'

# 3. Event Grouping (Detect time islands)
# Create a unique ID for each state change
df['event_id'] = (df['state_raw'] != df['state_raw'].shift()).cumsum()

# 4. Duration Filtering (The Magic)
# Calculate total duration of each event
event_durations = df.groupby('event_id')['delta_t'].transform('sum')

# Apply the Rule:
# If it's an Idle Candidate AND lasted less than the limit -> Becomes 'Working' (Maneuver/Short Pause)
# If it's an Idle Candidate AND lasted more than the limit -> Becomes 'Idle'
df['state'] = df['state_raw'] # Copy original

# Where it was Idle but too short -> Becomes Working (Maneuver)
mask_short_idle = (df['state_raw'] == 'Idle_Candidate') & (event_durations < MIN_IDLE_DURATION)
df.loc[mask_short_idle, 'state'] = 'Working' 

# Where it was Idle and long enough -> Confirms Idle
mask_real_idle = (df['state_raw'] == 'Idle_Candidate') & (event_durations >= MIN_IDLE_DURATION)
df.loc[mask_real_idle, 'state'] = 'Idle'

# 5. Final Consumption Calculation
df['fuel_idle'] = 0.0
df.loc[df['state'] == 'Idle', 'fuel_idle'] = df['liters_consumed']

# Transformation Report
n_raw = len(df[df['state_raw'] == 'Idle_Candidate'])
n_final = len(df[df['state'] == 'Idle'])
# Safety check for division by zero
pct_removed = 100 * (1 - n_final/n_raw) if n_raw > 0 else 0

print(f"Logic Applied: Speed < {SPEED_THRESHOLD*3.6:.1f} km/h AND Duration > {MIN_IDLE_DURATION}s")
print(f" - Total 'Slow' Records: {n_raw:,}")
print(f" - Final 'Idle' Records: {n_final:,}")
print(f" - {pct_removed:.1f}% of slow moments were reclassified as 'Maneuvers/Short Pauses'.")

In [ ]:
# Cell 7: Event-Based Classification (Hybrid: Speed + Duration)

print("--- Applying Event-Based Classification ---")

# 1. PARAMETERS (Your Logic)
SPEED_THRESHOLD = 0.75  # 2.7 km/h (More permissive on speed)
RPM_ON_THRESHOLD = 100
PTO_ACTIVE_THRESHOLD = 150
MIN_IDLE_DURATION = 0     # Seconds (Only counts as idle if stopped for > X sec)

# 2. Instant Classification (Idle Candidates)
# First, we mark everything that "looks like" idle
mask_off = df['engine_rpm'] < RPM_ON_THRESHOLD
mask_low_speed = df['veh_speed'] < SPEED_THRESHOLD
mask_pto_off = df['pto_rpm'].fillna(0) < PTO_ACTIVE_THRESHOLD

# Provisional State
df['state_raw'] = 'Working'
df.loc[mask_off, 'state_raw'] = 'Off'
df.loc[(~mask_off) & mask_low_speed & mask_pto_off, 'state_raw'] = 'Idle_Candidate'

# 3. Event Grouping (Detect time islands)
# Create a unique ID for each state change
df['event_id'] = (df['state_raw'] != df['state_raw'].shift()).cumsum()

# 4. Duration Filtering (The Magic)
# Calculate total duration of each event
event_durations = df.groupby('event_id')['delta_t'].transform('sum')

# Apply the Rule:
# If it's an Idle Candidate AND lasted less than the limit -> Becomes 'Working' (Maneuver/Short Pause)
# If it's an Idle Candidate AND lasted more than the limit -> Becomes 'Idle'
df['state'] = df['state_raw'] # Copy original

# Where it was Idle but too short -> Becomes Working (Maneuver)
mask_short_idle = (df['state_raw'] == 'Idle_Candidate') & (event_durations < MIN_IDLE_DURATION)
df.loc[mask_short_idle, 'state'] = 'Working' 

# Where it was Idle and long enough -> Confirms Idle
mask_real_idle = (df['state_raw'] == 'Idle_Candidate') & (event_durations >= MIN_IDLE_DURATION)
df.loc[mask_real_idle, 'state'] = 'Idle'

# 5. Final Consumption Calculation
df['fuel_idle'] = 0.0
df.loc[df['state'] == 'Idle', 'fuel_idle'] = df['liters_consumed']

# Transformation Report
n_raw = len(df[df['state_raw'] == 'Idle_Candidate'])
n_final = len(df[df['state'] == 'Idle'])
# Safety check for division by zero
pct_removed = 100 * (1 - n_final/n_raw) if n_raw > 0 else 0

print(f"Logic Applied: Speed < {SPEED_THRESHOLD*3.6:.1f} km/h AND Duration > {MIN_IDLE_DURATION}s")
print(f" - Total 'Slow' Records: {n_raw:,}")
print(f" - Final 'Idle' Records: {n_final:,}")
print(f" - {pct_removed:.1f}% of slow moments were reclassified as 'Maneuvers/Short Pauses'.")

In [ ]:
# Cell 9: Visualização Corrigida (100% PT-BR e Layout Compacto)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 0. PREPARAÇÃO E TRADUÇÃO
# Primeiro, traduzimos os estados dentro do DataFrame para garantir que a legenda fique em PT
# Mapeamento de Tradução
traducao_map = {
    'Idle': 'Ocioso',
    'Working': 'Trabalhando',
    'Off': 'Desligado'
}

# Criamos uma cópia para não alterar o original drasticamente
df_viz = df.copy()
df_viz['estado_pt'] = df_viz['state'].map(traducao_map).fillna(df_viz['state'])

# Agrupamos pelo nome em PORTUGUÊS
state_summary = df_viz.groupby('estado_pt').agg({
    'delta_t': 'sum', 
    'liters_consumed': 'sum'
}).reset_index()

# --- MAPA DE CORES FIXO (Chaves em Português) ---
COLOR_MAP = {
    'Ocioso': '#d62728',      # Vermelho
    'Trabalhando': '#2ca02c', # Verde
    'Desligado': '#d3d3d3'    # Cinza Claro
}

# Configuração dos Subplots (Aproximando os gráficos)
fig = make_subplots(
    rows=1, cols=2, 
    specs=[[{'type':'domain'}, {'type':'domain'}]],
    subplot_titles=("<b>Distribuição do Tempo</b>", "<b>Distribuição do Diesel</b>")
)

# --- Gráfico 1: Tempo (Inclui Desligado) ---
time_labels = state_summary['estado_pt']
time_values = state_summary['delta_t']
# Garante a cor correta para cada fatia
time_colors = [COLOR_MAP[x] for x in time_labels] 

fig.add_trace(go.Pie(
    labels=time_labels, 
    values=time_values,
    name="Tempo",
    marker_colors=time_colors,
    hole=.5,
    sort=False,
    showlegend=True           # Mostra legenda apenas neste (para não duplicar)
), 1, 1)

# --- Gráfico 2: Combustível (Apenas Trabalhando vs Ocioso) ---
# Filtramos 'Desligado' pois consumo é zero ou desprezível
fuel_summary = state_summary[state_summary['estado_pt'] != 'Desligado']
fuel_labels = fuel_summary['estado_pt']
fuel_values = fuel_summary['liters_consumed']
fuel_colors = [COLOR_MAP[x] for x in fuel_labels]

fig.add_trace(go.Pie(
    labels=fuel_labels, 
    values=fuel_values,
    name="Diesel",
    marker_colors=fuel_colors,
    hole=.5,
    sort=False,
    showlegend=False # Oculta legenda duplicada (usa a do primeiro gráfico)
), 1, 2)

# --- LAYOUT COMPACTO (A MÁGICA ACONTECE AQUI) ---
fig.update_layout(
    title_text="<b>Eficiência Operacional:</b> Vermelho indica desperdício",
    title_x=0.5, # Centraliza o título principal
    
    # Dimensões fixas para forçar a proximidade (proporção 2:1 compacta)
    width=800,  
    height=450,
    
    # Margens apertadas para remover espaço branco inútil
    margin=dict(t=80, b=30, l=40, r=40),
    
    # Legenda horizontal centralizada embaixo (evita esticar o gráfico para os lados)
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
    template='plotly_white' # Fundo branco limpo
)

fig.show()

In [ ]:
# Cell 10: Idle Detail (Who and Where?)

# --- PART 1: RANKING BY TRACTOR (The missing chart) ---
# Filter only idle data
tractor_idle = df[df['state'] == 'Idle'].groupby('tractor').agg({
    'delta_t': 'sum',
    'liters_consumed': 'sum'
}).reset_index()

tractor_idle['Idle Hours'] = tractor_idle['delta_t'] / 3600
# Sort from biggest villain to smallest (for the chart logic)
tractor_idle = tractor_idle.sort_values('Idle Hours', ascending=True)

fig1 = px.bar(
    tractor_idle, 
    x='Idle Hours', 
    y='tractor', 
    orientation='h',
    text='Idle Hours',
    color='liters_consumed',
    color_continuous_scale='Reds', # Red to indicate alert
    title="<b>Idle Ranking:</b> Which tractors stood still the longest?",
    labels={'liters_consumed': 'Waste (L)', 'tractor': 'Tractor'}
)
fig1.update_traces(texttemplate='%{text:.1f} h', textposition='outside')
fig1.update_layout(template=TEMPLATE, height=400)
fig1.show()

# --- PART 2: WASTE MATRIX (Heatmap) ---
matrix_data = df[df['state'] == 'Idle'].groupby(['tractor', 'activity'])['liters_consumed'].sum().reset_index()
heatmap_data = matrix_data.pivot(index='activity', columns='tractor', values='liters_consumed').fillna(0)

fig2 = px.imshow(
    heatmap_data,
    labels=dict(x="Tractor", y="Activity", color="Wasted Liters"),
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale='Magma',
    title="<b>Waste Matrix:</b> In which activity was diesel burned?"
)
fig2.update_layout(height=600, template=TEMPLATE)
fig2.show()

In [ ]:
# Cell 11: Financial and Environmental Impact Estimation

# --- PARAMETERS ---
DIESEL_PRICE = 1.588      # €/L or $/L (Adjust as needed)
EMISSION_FACTOR = 2.64    # kg CO2/L

# --- CALCULATIONS ---
# Calculating totals directly from df (since Cell 8 was skipped)
if 'df' in locals():
    idle_liters = df[df['state'] == 'Idle']['liters_consumed'].sum()
    total_liters = df['liters_consumed'].sum()
    pct_idle_diesel = (idle_liters / total_liters) * 100 if total_liters > 0 else 0
else:
    print("Warning: DataFrame 'df' not found. Ensure Cell 7 was run.")
    idle_liters = 0
    pct_idle_diesel = 0

# Financial and Environmental Math
wasted_cost = idle_liters * DIESEL_PRICE
avoidable_emissions = (idle_liters * EMISSION_FACTOR) / 1000 # Convert kg to Tonnes

# --- REPORT ---
print(f"=== STUDY CONCLUSION ===")
print(f"Detected Idleness:    {pct_idle_diesel:.1f}% of consumed diesel.")
print(f"Financial Impact:     € {wasted_cost:,.2f} wasted.")
print(f"Environmental Impact: {avoidable_emissions:.2f} tonnes of CO2 emitted needlessly.")

In [ ]:
# Cell 12: Perfil Detalhado por Trator (Storytelling & Visualização)

import plotly.express as px
import pandas as pd

# 0. MAPA DE TRADUÇÃO (Para garantir 100% PT)
traducoes_ativ = {
    'Transport': 'Transporte', 
    'Seed drill combination': 'Semeadura', 
    'Ploughing': 'Aração', 
    'not working': 'Não Trabalhando (Geral)',
    'Spraying': 'Pulverização', 
    'Fertilizing': 'Adubação',
    'Cultivating (deep)': 'Cultivo Profundo', 
    'Harvesting': 'Colheita',
    'Mowing (front)': 'Ceifa Frontal',
    'Mowing (large-scale)': 'Ceifa Larga',
    'Mulching': 'Trituração',
    'Power harrowing': 'Grade Rotativa',
    'Seedbed combination': 'Prep. Leito',
    'Cultivating (shallow)': 'Cultivo Superficial',
    'Disc harrowing': 'Gradeação',
    'Rotary tilling': 'Enxada Rotativa',
    'Precision air seeding': 'Semeadura Precisão'
}

# 1. PREPARAÇÃO DOS DADOS
# Traduzimos a coluna de atividade antes de agrupar
df_proc = df.copy()
df_proc['Atividade'] = df_proc['activity'].map(traducoes_ativ).fillna(df_proc['activity'])

# Agrupa por Trator e Atividade (PT)
profile = df_proc.groupby(['tractor', 'Atividade']).agg({
    'delta_t': 'sum',
    'liters_consumed': 'sum'
}).reset_index()

# Calcula o Desperdício (Ocioso) especificamente
# Nota: Verifica se o estado é 'Idle' (original) ou 'Ocioso' (se já foi traduzido antes)
profile_idle = df_proc[df_proc['state'].isin(['Idle', 'Ocioso'])].groupby(['tractor', 'Atividade']).agg({
    'delta_t': 'sum',
    'liters_consumed': 'sum'
}).reset_index().rename(columns={'delta_t': 'tempo_ocioso', 'liters_consumed': 'litros_ociosos'})

# Junta as tabelas
profile = pd.merge(profile, profile_idle, on=['tractor', 'Atividade'], how='left').fillna(0)

# Cálculos Percentuais e Renomeação para Boniteza do Gráfico
profile['% Desperdício'] = (profile['litros_ociosos'] / profile['liters_consumed']) * 100
profile['Horas Totais'] = profile['delta_t'] / 3600
profile['Consumo Total (L)'] = profile['liters_consumed'] # Nome bonito para o gráfico

# 2. STORYTELLING AUTOMATIZADO (Gera o texto para sua discussão)
print("="*80)
print("PERFIL INDIVIDUAL DA FROTA (Análise Granular)")
print("="*80)

tractors = profile['tractor'].unique()
for tractor in tractors:
    tractor_data = profile[profile['tractor'] == tractor]
    
    # Atividade que mais consumiu diesel
    top_diesel = tractor_data.sort_values('Consumo Total (L)', ascending=False).iloc[0]
    
    # Atividade com maior desperdício absoluto (Litros jogados fora)
    top_waste = tractor_data.sort_values('litros_ociosos', ascending=False).iloc[0]
    
    total_l_tractor = tractor_data['Consumo Total (L)'].sum()
    total_l_idle = tractor_data['litros_ociosos'].sum()
    pct_idle_tractor = (total_l_idle / total_l_tractor) * 100
    
    print(f"\n🚜 {tractor.upper()}")
    print(f"   - Consumo Total:        {total_l_tractor:,.1f} L")
    print(f"   - Desperdício Global:   {total_l_idle:,.1f} L ({pct_idle_tractor:.1f}%)")
    print(f"   - Atividade Principal:  {top_diesel['Atividade']} (Responsável por {(top_diesel['Consumo Total (L)']/total_l_tractor)*100:.1f}% do consumo)")
    print(f"   - Principal Gargalo:    {top_waste['Atividade']}")
    print(f"     (Nesta atividade, {top_waste['% Desperdício']:.1f}% do diesel foi queimado com a máquina parada)")

# 3. VISUALIZAÇÃO HIERÁRQUICA (Sunburst / Explosão Solar)
# Mostra: Quem (Centro) -> Fez o quê (Meio) -> Com que ineficiência (Cor)
fig = px.sunburst(
    profile,
    path=['tractor', 'Atividade'],  # Hierarquia: Trator -> Atividade
    values='Consumo Total (L)',     # Tamanho da fatia = Consumo
    color='% Desperdício',          # Cor da fatia = Eficiência
    
    # Escala: Verde (0% desperdício) -> Amarelo -> Vermelho (Alto desperdício)
    color_continuous_scale='RdYlGn_r', 
    range_color=[0, 15], # Trava o vermelho em 50% para não distorcer se houver outliers de 100%
    
    title="<b>Raio-X da Eficiência:</b> Tamanho = Consumo Total | Cor = % Desperdício",
    
    # Configura o que aparece ao passar o mouse (Hover)
    hover_data={
        'Horas Totais':':.1f', 
        'litros_ociosos':':.1f',
        'Consumo Total (L)': ':.1f',
        '% Desperdício': ':.1f'
    },
    labels={'litros_ociosos': 'Litros Desperdiçados'} # Tradução final de labels técnicos
)

# AJUSTE DE LAYOUT (O "ZOOM" SOLICITADO)
fig.update_layout(
    height=700, # Aumenta a altura para o círculo crescer
    margin=dict(t=50, l=0, r=0, b=0), # Remove bordas brancas laterais (Zoom efetivo)
    template='plotly_white',
    font=dict(size=14) # Aumenta a fonte para facilitar leitura
)

fig.show()

In [ ]:
# Cell 13: Formatted Table Generation for Word
summary_table = []

for tractor in profile['tractor'].unique():
    tractor_data = profile[profile['tractor'] == tractor]
    
    # Calculations
    total_l = tractor_data['liters_consumed'].sum()
    total_idle = tractor_data['litros_ociosos'].sum()
    pct_idle = (total_idle / total_l) * 100
    
    # Top Activity
    top_act = tractor_data.sort_values('liters_consumed', ascending=False).iloc[0]
    pct_top = (top_act['liters_consumed'] / total_l) * 100
    
    # Top Bottleneck
    top_bottleneck = tractor_data.sort_values('litros_ociosos', ascending=False).iloc[0]
    
    summary_table.append({
        'Tractor': tractor,
        'Total Consumption (L)': f"{total_l:,.1f}",
        'Global Waste': f"{total_idle:,.1f} L ({pct_idle:.1f}%)",
        'Main Activity': f"{top_act['Atividade']} ({pct_top:.1f}%)",
        'Biggest Bottleneck': f"{top_bottleneck['Atividade']} ({top_bottleneck['% Desperdício']:.1f}% stopped)"
    })

df_summary = pd.DataFrame(summary_table)
display(df_summary)

In [ ]:
# Cell 15: App Export (CORRECTED)

# 1. Create a clean copy
df_app = df.copy()

# 2. NAME CONFLICT RESOLUTION
# The App uses gauges (speedometers), so it needs the instantaneous rate (L/h).
# The 'liters_consumed' column we created in the notebook is the accumulated volume (Liters).
# We will drop the accumulated one to use the name 'liters_consumed' for the rate (L/h),
# maintaining compatibility with your App's code.
if 'liters_consumed' in df_app.columns:
    df_app = df_app.drop(columns=['liters_consumed'])

# 3. Rename original sensor columns
# Adjust the keys on the left ('veh_speed', etc.) if the name in your df is different
mapping = {
    'veh_speed': 'speed',          # Speed
    'engine_rpm': 'engine_speed',  # RPM
    'fuel_rate': 'liters_consumed' # Now becomes L/h (no conflict)
}

df_app = df_app.rename(columns=mapping)

# 4. Column Selection (To keep the file light and clean)
final_columns = [
    'timestamp', 'tractor', 'latitude', 'longitude', 
    'speed', 'engine_speed', 'liters_consumed', 
    'activity', 'state'
]

# Filter only what exists (to avoid errors if something is missing)
existing_cols = [c for c in final_columns if c in df_app.columns]
df_app = df_app[existing_cols]

# 5. Save the File
app_path = PROJECT_ROOT / "data" / "processed" / "telemetry_app.parquet"
print(f"Saving data for the App at: {app_path}")

# Ensure no residual duplicates
df_app = df_app.loc[:, ~df_app.columns.duplicated()]

df_app.to_parquet(app_path)
print("✅ Success! The file 'telemetry_app.parquet' was generated correctly.")

In [ ]:
import pandas as pd
path = r'E:\Tese\idletime\data\processed\telemetry_app.parquet'
df = pd.read_parquet(path)  # requires pandas + pyarrow or fastparquet
print(df.columns.tolist())